# 04 — Embedding Verification

**Primary author:** Victoria

**Builds on:**
- *03_train_g1.ipynb* (Victoria — triplet construction that produced the g_1 model whose embeddings we are verifying)
- *scripts/embed_val.py* (Victoria / Nathan — the Great Lakes embedding script that produced the four `.npy` sets being checked)
- *DECISIONS.md §20* (mean pooling is canonical; the `_tokenspan` suffix marks the non-canonical extraction method)

**Prompt engineering:** Victoria  
**AI assistance:** Claude / Claude Code (Anthropic)  
**Environment:** Local

---

This notebook is a pre-Stage-5 verification gate. During Stage 3–4 the
original g_1 model (token-span extraction) was renamed to `g1_tokenspan`,
and a new `g1` was trained with mean pooling per Decision 20. Embeddings
were then regenerated on Great Lakes for all four `(model, pooling)`
combinations. Before we start hypothesis testing in Stage 5, we want
systematic evidence that nothing was swapped, duplicated, or mislabeled
during the rename and regeneration.

The check has two parts:

1. **Per-file integrity** (§2) — every `.npy` has the expected shape, no
   NaN, no all-zero rows, and a plausible L2-norm fingerprint. The four
   `f_clue_val_index.csv` files should carry identical (clue_id, definition)
   columns because they all describe the same validation-split phrase file.

2. **Cross-model rowwise cosine similarity** (§3–§5) — for each of the three
   phrase types we compute a 4×4 matrix of mean rowwise cosine similarities.
   The *pattern* of divergences is the diagnostic: pairs that differ only in
   pooling should look different from pairs that differ only in weights, and
   any off-diagonal cell ≥ 0.999 would indicate two files that are
   effectively the same (a bad sign).

This is a CPU-only notebook — it loads existing `.npy` files and computes
summary statistics. No new embeddings are generated and nothing under
`data/` is written.

**Reads:**
- `data/embeddings/{g_stock, g_stock_tokenspan, g1, g1_tokenspan}/f_clue_val.npy` and `f_clue_val_index.csv`
- `data/embeddings/{g_stock, g_stock_tokenspan, g1, g1_tokenspan}/f_common_wndef_val.npy`
- `data/embeddings/{g_stock, g_stock_tokenspan, g1, g1_tokenspan}/f_common_wnex_val.npy`
- `data/filtered_split/wn_synset/wndef/vocabulary_wndef_val.csv`
- `data/filtered_split/wn_synset/wnex/vocabulary_wnex_val.csv`

**Writes:**
- `outputs/04_embedding_verification-results.md` — PASS/FAIL verdict and the full set of summary tables

---

## §1 — Setup and file loading

Environment auto-detection lets this notebook run unmodified on Local,
Great Lakes, and Colab. All the work here is `numpy` and `pandas` — there
is no GPU dependency.

We load all 12 `.npy` arrays into a dict keyed by `(model_name, phrase_type)`,
plus the four `f_clue_val_index.csv` files (needed for §2's index-consistency
check) and the two vocabulary files that index the `f_common_*_val.npy`
arrays (needed for shape validation in §2).

In [ ]:
# ============================================================
# Imports and configuration
# ============================================================
import time
from datetime import date
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

# --- Environment auto-detection ---
try:
    IS_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IS_COLAB = False

IS_GREATLAKES = Path("/nfs/turbo").exists()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Research Project - NLP CCC's/ccc-project")
elif IS_GREATLAKES:
    PROJECT_ROOT = Path.home() / "ccc-project"
else:
    PROJECT_ROOT = Path("../..").resolve()

COMPONENT_ROOT  = PROJECT_ROOT / "custom_embedding_model"
EMBEDDINGS_DIR  = COMPONENT_ROOT / "data" / "embeddings"
WN_DIR          = COMPONENT_ROOT / "data" / "filtered_split" / "wn_synset"
OUTPUT_DIR      = COMPONENT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

env_label = "Colab" if IS_COLAB else ("Great Lakes" if IS_GREATLAKES else "Local")
print(f"Environment:    {env_label}")
print(f"PROJECT_ROOT:   {PROJECT_ROOT}")
print(f"EMBEDDINGS_DIR: {EMBEDDINGS_DIR}")
print(f"OUTPUT_DIR:     {OUTPUT_DIR}")

# Reproducibility — no randomness in this notebook, but pin anyway so that
# any numpy utilities behave identically across re-runs.
np.random.seed(42)

# Version reporting (Decision 18).
print(f"\npandas:  {pd.__version__}")
print(f"numpy:   {np.__version__}")

In [ ]:
# ============================================================
# Model and phrase-type registry
# ============================================================
# MODEL_NAMES are ordered (stock, tokenspan-stock, g1-meanpool, g1-tokenspan)
# so that the 4x4 similarity matrices read naturally:
#   - adjacent cells along the diagonal are "same weights, different pooling"
#   - top-left quadrant is g_stock-only, bottom-right is g1-only
MODEL_NAMES  = ["g_stock", "g_stock_tokenspan", "g1", "g1_tokenspan"]
PHRASE_TYPES = ["f_clue_val", "f_common_wndef_val", "f_common_wnex_val"]

# Expected shapes from FINDINGS.md — these are the row counts produced by
# Stage 2 phrase construction on the validation split, and by embed_val.py
# over the validation f_clue phrase file.
EXPECTED_SHAPE = {
    "f_clue_val":         (47933, 1024),
    "f_common_wndef_val": (26152, 1024),
    "f_common_wnex_val":  (3008, 1024),
}

In [ ]:
# ============================================================
# Load all 12 .npy arrays and the 4 f_clue_val index CSVs
# ============================================================
t0 = time.time()

# embeddings: (model, phrase) -> np.ndarray of shape (N, 1024)
embeddings = {}
for model in MODEL_NAMES:
    for phrase in PHRASE_TYPES:
        path = EMBEDDINGS_DIR / model / f"{phrase}.npy"
        embeddings[(model, phrase)] = np.load(path)

# f_clue_val_index.csv — keep_default_na=False because definition strings
# may legitimately be "nan" (grandmother) and we never want them coerced.
clue_indexes = {}
for model in MODEL_NAMES:
    clue_indexes[model] = pd.read_csv(
        EMBEDDINGS_DIR / model / "f_clue_val_index.csv",
        keep_default_na=False,
        na_values=[""],
    )

# Vocabulary files that index the f_common_*_val.npy arrays.
vocab_wndef_val = pd.read_csv(
    WN_DIR / "wndef" / "vocabulary_wndef_val.csv",
    keep_default_na=False, na_values=[""],
)
vocab_wnex_val  = pd.read_csv(
    WN_DIR / "wnex" / "vocabulary_wnex_val.csv",
    keep_default_na=False, na_values=[""],
)

load_seconds = time.time() - t0
print(f"Loaded 12 .npy files + 4 index CSVs + 2 vocab CSVs in {load_seconds:.1f}s")
print(f"\nvocabulary_wndef_val.csv: {len(vocab_wndef_val):,} rows")
print(f"vocabulary_wnex_val.csv:  {len(vocab_wnex_val):,} rows")
print(f"\nf_clue_val_index.csv row counts (should all agree):")
for model in MODEL_NAMES:
    print(f"  {model:22s}: {len(clue_indexes[model]):,}")

---

## §2 — Shape and integrity checks

Five checks run over every `.npy` file plus one check across the four
`f_clue_val_index.csv` files:

1. **Shape** matches the expected `(rows, 1024)` from FINDINGS.md — catches
   any file that was generated against the wrong phrase subset.
2. **No NaN** — an NaN anywhere in an embedding would silently poison every
   cosine computed against it.
3. **No all-zero rows** — a zero row means the encoder returned an unusable
   vector for that phrase (e.g., empty input, or a tokenization failure).
   We compute L2 norms and flag any that are exactly zero.
4. **L2-norm fingerprint** — min / mean / max norm. Different models and
   different pooling strategies produce different norm distributions, so
   two files that share a fingerprint to high precision are suspicious.
5. **Index consistency** — all four `f_clue_val_index.csv` files should
   carry the same (clue_id, definition) pairs in the same order, because
   every model was embedded against the same validation phrase file. We
   use `DataFrame.equals()` on those two columns as the equality check.

In [ ]:
# ============================================================
# Per-file integrity table
# ============================================================
records = []
for model in MODEL_NAMES:
    for phrase in PHRASE_TYPES:
        emb = embeddings[(model, phrase)]
        # L2 norm per row — used both for the zero-row check and the fingerprint.
        norms = np.linalg.norm(emb, axis=1)
        records.append({
            "model":      model,
            "phrase":     phrase,
            "shape":      tuple(emb.shape),
            "shape_ok":   tuple(emb.shape) == EXPECTED_SHAPE[phrase],
            "has_nan":    bool(np.isnan(emb).any()),
            "n_zero":     int((norms == 0).sum()),
            "L2_min":     float(norms.min()),
            "L2_mean":    float(norms.mean()),
            "L2_max":     float(norms.max()),
        })

integrity_df = pd.DataFrame(records)

# Pretty-print — float columns to 4 decimals for readability.
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.max_colwidth", 40,
                       "display.width", 140):
    print(integrity_df.to_string(index=False))

In [ ]:
# ============================================================
# Cross-check against vocabulary lengths
# ============================================================
# The f_common_*_val arrays must be indexed by their corresponding vocabulary
# file. Match these row counts to the vocabulary files regardless of whether
# the expected-shape check above passed — the shape check reads from a frozen
# constant, while this check reads from the actual vocabulary files on disk.
wndef_expected = len(vocab_wndef_val)
wnex_expected  = len(vocab_wnex_val)

for model in MODEL_NAMES:
    n_wndef = embeddings[(model, "f_common_wndef_val")].shape[0]
    n_wnex  = embeddings[(model, "f_common_wnex_val")].shape[0]
    print(f"{model:22s} f_common_wndef_val rows: {n_wndef:,} (vocab: {wndef_expected:,}) "
          f"{'OK' if n_wndef == wndef_expected else 'MISMATCH'}")
    print(f"{'':22s} f_common_wnex_val  rows: {n_wnex:,} (vocab: {wnex_expected:,}) "
          f"{'OK' if n_wnex == wnex_expected else 'MISMATCH'}")

In [ ]:
# ============================================================
# f_clue_val_index.csv consistency across models
# ============================================================
# All four embedding runs should have been given the same f_clue_val.csv
# phrase file, so the (clue_id, definition) columns must agree exactly in
# content AND order. Reset index so equals() is not confused by any residual
# index differences from the original reads.
ref_model = MODEL_NAMES[0]
ref_index = clue_indexes[ref_model][["clue_id", "definition"]].reset_index(drop=True)

index_consistency = {}
for model in MODEL_NAMES[1:]:
    other = clue_indexes[model][["clue_id", "definition"]].reset_index(drop=True)
    index_consistency[model] = bool(ref_index.equals(other))

print(f"Reference model: {ref_model} ({len(ref_index):,} rows)\n")
for model, ok in index_consistency.items():
    print(f"  {model:22s} index matches {ref_model}: {ok}")

all_indexes_match = all(index_consistency.values())

---

## §3 — Pairwise cosine similarity: f_clue_val

For each of the C(4, 2) = 6 pairs of models, we compute the rowwise cosine
similarity between their `f_clue_val.npy` arrays and report mean, median,
min, and std. The 4×4 matrix of means (symmetric, diagonal = 1.0) is the
summary view; the per-pair table exposes the distribution detail.

**Interpretation guide** (predictions against which to read the results):

- **Same weights, different pooling** (`g_stock` vs `g_stock_tokenspan`;
  `g1` vs `g1_tokenspan`) — expect moderate divergence. The consistency
  check in `embed_val.py` found ≈0.926 for `g_stock` f_clue on the full
  dataset; the validation subset may differ slightly.
- **Same pooling, different weights** (`g_stock` vs `g1`, both mean-pool;
  `g_stock_tokenspan` vs `g1_tokenspan`, both token-span) — expect
  relatively high similarity. Fine-tuning nudges weights, it does not
  overhaul them.
- **Different weights AND different pooling** — expect the lowest
  similarity of any pair, because both sources of divergence compound.
- **Red flag:** any off-diagonal cell ≥ 0.999 would indicate two files
  that are effectively identical — the specific failure mode this
  notebook exists to catch.

In [ ]:
# ============================================================
# Rowwise cosine similarity helpers
# ============================================================
def rowwise_cosine(A, B):
    """Return per-row cosine similarity between two equal-shape (N, D) arrays.

    Matches the convention used in DATA.md for ATE computation: normalize each
    row to unit length, then take the dotproduct.
    """
    # The +1e-10 guard handles zero rows if any slipped past the §2 check;
    # we already assert n_zero == 0 in the verdict, so this is belt-and-braces.
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-10)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-10)
    return np.sum(A_norm * B_norm, axis=1)

def pairwise_cosine_table(phrase):
    """Build a 4x4 mean-similarity matrix and a per-pair detail DataFrame."""
    mat = pd.DataFrame(
        np.ones((len(MODEL_NAMES), len(MODEL_NAMES))),  # diagonal = 1
        index=MODEL_NAMES, columns=MODEL_NAMES,
    )
    detail = []
    for a, b in combinations(MODEL_NAMES, 2):
        sims = rowwise_cosine(embeddings[(a, phrase)], embeddings[(b, phrase)])
        mean = float(sims.mean())
        mat.loc[a, b] = mean
        mat.loc[b, a] = mean
        detail.append({
            "model_a": a,
            "model_b": b,
            "mean":    mean,
            "median":  float(np.median(sims)),
            "min":     float(sims.min()),
            "std":     float(sims.std()),
        })
    return mat, pd.DataFrame(detail)

In [ ]:
# ============================================================
# f_clue_val: 4x4 mean matrix + 6-pair detail
# ============================================================
t0 = time.time()
mat_clue, detail_clue = pairwise_cosine_table("f_clue_val")
t_clue = time.time() - t0

print(f"f_clue_val — 4x4 mean rowwise cosine similarity "
      f"(N={embeddings[(MODEL_NAMES[0], 'f_clue_val')].shape[0]:,} rows per pair):\n")
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 140):
    print(mat_clue.to_string())

print(f"\nPer-pair detail (6 unique pairs):\n")
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 140):
    print(detail_clue.to_string(index=False))

print(f"\nCompute time: {t_clue:.1f}s")

---

## §4 — Pairwise cosine similarity: f_common_wndef_val

Same 4×4 matrix and 6-pair detail table, now over the 26,152 validation
`f_common_wndef_val.npy` rows. This phrase type is especially informative:
the triplet loss was trained on `f_common_wndef` phrases (positives and
negatives), so the effect of fine-tuning should be most visible here.
If `g1` vs `g_stock` looks close to 1.0 on `f_common_wndef_val`, the
training had almost no effect on the targeted format.

In [ ]:
# ============================================================
# f_common_wndef_val: 4x4 mean matrix + 6-pair detail
# ============================================================
t0 = time.time()
mat_wndef, detail_wndef = pairwise_cosine_table("f_common_wndef_val")
t_wndef = time.time() - t0

print(f"f_common_wndef_val — 4x4 mean rowwise cosine similarity "
      f"(N={embeddings[(MODEL_NAMES[0], 'f_common_wndef_val')].shape[0]:,} rows per pair):\n")
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 140):
    print(mat_wndef.to_string())

print(f"\nPer-pair detail (6 unique pairs):\n")
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 140):
    print(detail_wndef.to_string(index=False))

print(f"\nCompute time: {t_wndef:.1f}s")

---

## §5 — Pairwise cosine similarity: f_common_wnex_val

The same 4×4 matrix and 6-pair detail table, now over the 3,008 validation
`f_common_wnex_val.npy` rows.

This is the cross-f generalization question in preview form: the model was
trained on `wndef` phrases, not `wnex` phrases. If `g1` changes `wnex`
embeddings substantially (low cosine vs `g_stock`), that hints at semantic
generalization — the model learned something that transfers across phrase
formats. If `wnex` embeddings barely move (high cosine vs `g_stock`), that
hints at format-specific learning — the model memorized the layout of
`<t>word</t>: definition` rather than the concept.

This notebook records the observation but defers the full interpretation
to the Stage 5 hypothesis tests.

In [ ]:
# ============================================================
# f_common_wnex_val: 4x4 mean matrix + 6-pair detail
# ============================================================
t0 = time.time()
mat_wnex, detail_wnex = pairwise_cosine_table("f_common_wnex_val")
t_wnex = time.time() - t0

print(f"f_common_wnex_val — 4x4 mean rowwise cosine similarity "
      f"(N={embeddings[(MODEL_NAMES[0], 'f_common_wnex_val')].shape[0]:,} rows per pair):\n")
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 140):
    print(mat_wnex.to_string())

print(f"\nPer-pair detail (6 unique pairs):\n")
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 140):
    print(detail_wnex.to_string(index=False))

print(f"\nCompute time: {t_wnex:.1f}s")

---

## §6 — Overall verdict

We combine the §2 per-file checks with the §3–§5 pairwise matrices into a
single PASS/FAIL verdict against four criteria:

1. **Shapes** — every one of the 12 `.npy` files has the expected
   `(rows, 1024)` shape. FAIL otherwise.
2. **Integrity** — no NaN values and no all-zero rows anywhere. FAIL
   otherwise.
3. **No accidental duplicates** — every off-diagonal mean cosine across
   all three 4×4 matrices is < 0.999. FAIL otherwise, because two files
   at that similarity are effectively the same embeddings despite
   different names.
4. **Internal consistency of the divergence pattern** — pairs differing
   only in pooling should look similar to each other, and pairs differing
   only in weights should look similar to each other. This is a WARNING
   rather than a FAIL: the pattern can be surprising for legitimate
   reasons and should be flagged for human review, not auto-rejected.

We also carry the §2 index-consistency check forward as its own criterion,
since a mismatch would invalidate every row-aligned comparison downstream.

In [ ]:
# ============================================================
# Verdict logic
# ============================================================
all_shapes_ok      = bool(integrity_df["shape_ok"].all())
no_nan             = bool(~integrity_df["has_nan"].any())
no_zero_rows       = bool((integrity_df["n_zero"] == 0).all())

# Collect every off-diagonal mean across all three matrices.
off_diag = []
for label, mat in [("f_clue_val", mat_clue),
                   ("f_common_wndef_val", mat_wndef),
                   ("f_common_wnex_val", mat_wnex)]:
    for a, b in combinations(MODEL_NAMES, 2):
        off_diag.append((label, a, b, mat.loc[a, b]))

off_diag_df = pd.DataFrame(off_diag, columns=["phrase", "model_a", "model_b", "mean_cos"])
max_off_diag = float(off_diag_df["mean_cos"].max())
dup_pairs    = off_diag_df[off_diag_df["mean_cos"] >= 0.999]
no_duplicates = len(dup_pairs) == 0

# Pattern-consistency heuristic: for each phrase type, check that the spread
# among "same weights / different pooling" pairs is small, same for
# "different weights / same pooling" pairs. "Small" = std of pair means
# within the category below a threshold we hand-pick (0.05 is loose enough
# to tolerate genuine variation while flagging an odd outlier).
same_weights_diff_pool = [("g_stock", "g_stock_tokenspan"), ("g1", "g1_tokenspan")]
same_pool_diff_weights = [("g_stock", "g1"), ("g_stock_tokenspan", "g1_tokenspan")]

pattern_warnings = []
for label, mat in [("f_clue_val", mat_clue),
                   ("f_common_wndef_val", mat_wndef),
                   ("f_common_wnex_val", mat_wnex)]:
    sw = [mat.loc[a, b] for a, b in same_weights_diff_pool]
    sp = [mat.loc[a, b] for a, b in same_pool_diff_weights]
    sw_spread = float(np.std(sw))
    sp_spread = float(np.std(sp))
    if sw_spread > 0.05:
        pattern_warnings.append(
            f"{label}: same-weights/diff-pool pairs disagree (std={sw_spread:.4f}) — "
            f"{same_weights_diff_pool[0]}={sw[0]:.4f} vs {same_weights_diff_pool[1]}={sw[1]:.4f}"
        )
    if sp_spread > 0.05:
        pattern_warnings.append(
            f"{label}: same-pool/diff-weights pairs disagree (std={sp_spread:.4f}) — "
            f"{same_pool_diff_weights[0]}={sp[0]:.4f} vs {same_pool_diff_weights[1]}={sp[1]:.4f}"
        )

# Combine criteria.
criteria = {
    "All shapes match expected":     all_shapes_ok,
    "No NaN anywhere":               no_nan,
    "No all-zero rows":              no_zero_rows,
    "f_clue indexes consistent":     all_indexes_match,
    "No off-diagonal cell >= 0.999": no_duplicates,
}
verdict = "PASS" if all(criteria.values()) else "FAIL"

print(f"VERDICT: {verdict}\n")
print("Criteria:")
for name, ok in criteria.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
print(f"\nMax off-diagonal mean cosine (across all three matrices): {max_off_diag:.4f}")

if pattern_warnings:
    print("\nWARNINGS (pattern inconsistencies — human review recommended):")
    for w in pattern_warnings:
        print(f"  - {w}")
else:
    print("\nNo pattern-consistency warnings.")

---

## §7 — Write results file

We write a markdown results file containing the versions, the integrity
table, the three 4×4 matrices, the six-pair detail tables, the verdict,
and any pattern warnings. This file is the permanent record of the check
and is what the Architect reads when reviewing Stage 4.

In [ ]:
# ============================================================
# Build and write outputs/04_embedding_verification-results.md
# ============================================================
def _fmt_cell(v, float_fmt="{:.4f}"):
    # Numeric values render at fixed precision; everything else stringifies.
    if isinstance(v, (float, np.floating)):
        return float_fmt.format(v)
    return str(v)

def df_to_md(df, float_fmt="{:.4f}"):
    """Render a DataFrame as a GitHub-flavored pipe-delimited markdown table.

    Hand-rolled to avoid a hard dependency on the optional tabulate package.
    """
    cols = list(df.columns)
    header = "| " + " | ".join(cols) + " |"
    sep    = "|" + "|".join(["---"] * len(cols)) + "|"
    rows = [
        "| " + " | ".join(_fmt_cell(v, float_fmt) for v in row) + " |"
        for row in df.itertuples(index=False, name=None)
    ]
    return chr(10).join([header, sep, *rows])

def mat_to_md(mat):
    """Render a 4x4 similarity matrix with the row label as the first column."""
    return df_to_md(mat.reset_index().rename(columns={"index": "model"}))

lines = []
lines.append("# Results: 04 — Embedding Verification\n")
lines.append(f"**Date:** {date.today().isoformat()}  ")
lines.append(f"**Environment:** {env_label}\n")

lines.append("## Versions\n")
lines.append(f"- pandas: {pd.__version__}")
lines.append(f"- numpy:  {np.__version__}\n")

lines.append("## Overall verdict\n")
lines.append(f"**{verdict}**\n")
lines.append("| Criterion | Result |")
lines.append("|---|---|")
for name, ok in criteria.items():
    lines.append(f"| {name} | {'PASS' if ok else 'FAIL'} |")
lines.append("")
lines.append(f"Max off-diagonal mean cosine across all three matrices: **{max_off_diag:.4f}**\n")

if pattern_warnings:
    lines.append("### Pattern-consistency warnings\n")
    for w in pattern_warnings:
        lines.append(f"- {w}")
    lines.append("")
else:
    lines.append("No pattern-consistency warnings.\n")

lines.append("## §2 — Per-file integrity\n")
# Convert shape tuple to a string for markdown rendering (tabulate can't format tuples nicely).
integrity_md = integrity_df.copy()
integrity_md["shape"] = integrity_md["shape"].apply(lambda t: f"{t[0]} x {t[1]}")
lines.append(df_to_md(integrity_md))
lines.append("")

lines.append("### f_clue_val index consistency\n")
lines.append(f"Reference: `{ref_model}` ({len(ref_index):,} rows)\n")
lines.append("| Other model | Matches reference |")
lines.append("|---|---|")
for model, ok in index_consistency.items():
    lines.append(f"| {model} | {ok} |")
lines.append("")

lines.append("## §3 — f_clue_val pairwise cosine (mean)\n")
lines.append(mat_to_md(mat_clue))
lines.append("")
lines.append("### Per-pair detail\n")
lines.append(df_to_md(detail_clue))
lines.append("")

lines.append("## §4 — f_common_wndef_val pairwise cosine (mean)\n")
lines.append(mat_to_md(mat_wndef))
lines.append("")
lines.append("### Per-pair detail\n")
lines.append(df_to_md(detail_wndef))
lines.append("")

lines.append("## §5 — f_common_wnex_val pairwise cosine (mean)\n")
lines.append(mat_to_md(mat_wnex))
lines.append("")
lines.append("### Per-pair detail\n")
lines.append(df_to_md(detail_wnex))
lines.append("")

lines.append("## Runtime\n")
lines.append(f"- Load 12 .npy + 4 index CSV + 2 vocab CSV: {load_seconds:.1f}s")
lines.append(f"- §3 f_clue_val pairwise cosine:           {t_clue:.1f}s")
lines.append(f"- §4 f_common_wndef_val pairwise cosine:   {t_wndef:.1f}s")
lines.append(f"- §5 f_common_wnex_val pairwise cosine:    {t_wnex:.1f}s")
lines.append("")

results_path = OUTPUT_DIR / "04_embedding_verification-results.md"
results_path.write_text("\n".join(lines))
print(f"Wrote {results_path}")
print(f"Size: {results_path.stat().st_size / 1024:.1f} KB")

---

## Summary

This notebook verified the four sets of validation embeddings
(`g_stock`, `g_stock_tokenspan`, `g1`, `g1_tokenspan`) before Stage 5
hypothesis testing. It performed:

- **12 shape checks** against FINDINGS.md's expected `(47933, 1024)`,
  `(26152, 1024)`, `(3008, 1024)` — plus a separate cross-check against the
  two validation vocabulary files on disk.
- **12 NaN, zero-row, and L2-norm fingerprint checks** — one per `.npy`.
- **4-way f_clue index consistency** — confirming all four models were
  embedded against the same (clue_id, definition) phrase file.
- **Three 4×4 rowwise cosine matrices** (18 off-diagonal means) probing
  whether any two files are accidentally the same despite different names,
  and whether the `(pooling, weights)` divergence pattern is internally
  consistent.

**Verdict:** printed at the end of §6. PASS means all integrity checks and
the no-accidental-duplicates test succeeded; any pattern-consistency
warnings are shown separately for human review.

**Outputs:**
- `outputs/04_embedding_verification-results.md` — complete summary with
  tables, verdict, and any warnings.

**Runtime:** A few seconds on CPU — dominated by `.npy` loading and three
pairwise-cosine sweeps over the 12 arrays. No GPU required.